In [1]:
!pip install -q -U transformers bitsandbytes accelerate trl peft
!pip install -q -U rouge-score bert-score datasets

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [3]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from huggingface_hub import login

login()

MODEL_ID = "google/gemma-4-E4B-it"
SEED = 42

print(torch.cuda.get_device_name(0))
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


NVIDIA A100-SXM4-80GB
Total VRAM: 85.09 GB


In [4]:
print("Loading dataset...")
ds = load_dataset("Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset")

split1 = ds["train"].train_test_split(test_size=0.1, seed=SEED)
split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)

train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

Loading dataset...


Train: 47880
Val:   2660
Test:  2661


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    messages = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}


In [6]:
train_data_formatted = train_data.map(format_example)
val_data_formatted   = val_data.map(format_example)

def count_tokens(example):
    tokens = tokenizer(example["text"], return_tensors="pt")
    return {"token_count": tokens["input_ids"].shape[1]}

train_data_with_counts = train_data_formatted.map(count_tokens)
counts = train_data_with_counts["token_count"]

print(f"Min:             {np.min(counts)}")
print(f"Max:             {np.max(counts)}")
print(f"Mean:            {np.mean(counts):.0f}")
print(f"Median:          {np.median(counts):.0f}")
print(f"95th percentile: {np.percentile(counts, 95):.0f}")
print(f"Examples over 4096: {sum(1 for c in counts if c > 4096)}")

Min:             255
Max:             1274
Mean:            651
Median:          645
95th percentile: 782
Examples over 4096: 0


In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

print(f"VRAM after model load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

VRAM after model load: 16.32 GB


In [8]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

In [9]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)

<IPython.core.display.Javascript object>

In [10]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/cybersec-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    learning_rate=2e-4,
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
)

tokenizer.model_max_length = 1536

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data_formatted,
    eval_dataset=val_data_formatted,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"VRAM before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
trainer.train(resume_from_checkpoint="/content/drive/MyDrive/cybersec-finetuned/checkpoint-1400")

# Save final model
trainer.save_model("/content/drive/MyDrive/cybersec-finetuned")
print("Training complete. Model saved.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


VRAM before training: 16.42 GB


Step,Training Loss,Validation Loss
1600,0.558400,0.575051
1800,0.562152,0.568580
2000,0.571979,0.562330
2200,0.558521,0.557004
2400,0.555368,0.552465
2600,0.542025,0.548481
2800,0.536145,0.545609
2993,0.553253,0.544022


Training complete. Model saved.


In [11]:
MAX_SAMPLES    = 50
MAX_NEW_TOKENS = 128

def build_prompt(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user",   "content": example["user"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

rows       = test_data.select(range(min(MAX_SAMPLES, len(test_data))))
references = [r["assistant"].strip() for r in rows]

print(f"\nLoading base {MODEL_ID} for inference...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.eval()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

base_predictions = []
for i, ex in enumerate(rows):
    prompt = build_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(base_model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = base_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    base_predictions.append(pred)
    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(rows)}] input: {ex['user'][:60]}...")
        print(f"           output: {pred[:60]}...")

del base_model
torch.cuda.empty_cache()


Loading base google/gemma-4-E4B-it for inference...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

VRAM used: 32.73 GB
[1/50] input: In what situations would adversary emulation reveal gaps in ...
           output: Adversary emulation is a powerful technique for stress-testi...
[10/50] input: Which situation-specific threat models best capture the atta...
           output: This is a sophisticated question that bridges threat modelin...
[20/50] input: What are the techniques to detect and recover shared memory ...
           output: ## Detecting and Recovering Shared Memory Segments: A Cyber-...
[30/50] input: What are the techniques for developing purple team methodolo...
           output: ## Purple Teaming for Software Supply Chain Security: Execut...
[40/50] input: In which circumstances could deception technologies (honeypo...
           output: Deception technologies are highly decisive in the early dete...
[50/50] input: What techniques can defenders use to correlate campaigns thr...
           output: ## Correlating Campaigns via Victimology Patterns: Executive...


In [14]:
!pip install -q torchao --upgrade
import importlib
import peft
importlib.reload(peft)
from peft import PeftModel

print(f"\nLoading fine-tuned model for inference...")
ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(ft_base, "/content/drive/MyDrive/cybersec-finetuned")
ft_model.eval()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

ft_predictions = []
for i, ex in enumerate(rows):
    prompt = build_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(ft_model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    ft_predictions.append(pred)
    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(rows)}] input: {ex['user'][:60]}...")
        print(f"           output: {pred[:60]}...")

del ft_model
torch.cuda.empty_cache()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.2 MB/s eta 0:00:00

Loading fine-tuned model for inference...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

VRAM used: 48.82 GB
[1/50] input: In what situations would adversary emulation reveal gaps in ...
           output: Adversary emulation reveals gaps in threat intelligence when...
[10/50] input: Which situation-specific threat models best capture the atta...
           output: Intelligence Analyst Integration (IAI) in Hunt Teams introdu...
[20/50] input: What are the techniques to detect and recover shared memory ...
           output: Detecting and recovering shared memory segments requires a m...
[30/50] input: What are the techniques for developing purple team methodolo...
           output: Purple team methodologies for software supply chain security...
[40/50] input: In which circumstances could deception technologies (honeypo...
           output: Deception technologies can be decisive in early detection of...
[50/50] input: What techniques can defenders use to correlate campaigns thr...
           output: Defenders can employ several sophisticated techniques to cor...


In [15]:
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

base_rouge = [rouge.score(ref, pred)["rougeL"].fmeasure
              for ref, pred in zip(references, base_predictions)]
ft_rouge   = [rouge.score(ref, pred)["rougeL"].fmeasure
              for ref, pred in zip(references, ft_predictions)]

print(f"ROUGE-L  | Base: {sum(base_rouge)/len(base_rouge):.4f} | Fine-tuned: {sum(ft_rouge)/len(ft_rouge):.4f}")

_, _, base_bert = bert_score_fn(base_predictions, references,
                                lang="en", model_type="roberta-large",
                                device="cpu", use_fast_tokenizer=False, verbose=False)
_, _, ft_bert   = bert_score_fn(ft_predictions, references,
                                lang="en", model_type="roberta-large",
                                device="cpu", use_fast_tokenizer=False, verbose=False)

print(f"BERTScore| Base: {base_bert.mean().item():.4f} | Fine-tuned: {ft_bert.mean().item():.4f}")

ROUGE-L  | Base: 0.1284 | Fine-tuned: 0.1857


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore| Base: 0.8284 | Fine-tuned: 0.8628


In [16]:
!pip install openai

from openai import OpenAI
import json
import time
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

JUDGE_PROMPT = """You are a cybersecurity expert evaluating the quality of AI-generated responses to security questions.

Rate the following response on a scale of 1 to 5 using these criteria:
- 5: Technically accurate, complete, well-structured, and directly addresses the question
- 4: Mostly accurate and complete with minor gaps
- 3: Partially correct but missing important details or contains minor inaccuracies
- 2: Superficial or significantly incomplete with notable inaccuracies
- 1: Incorrect, irrelevant, or dangerously misleading

Return ONLY a JSON object with no preamble or markdown. Example:
{{"score": 4, "reason": "Response correctly identifies the attack vector but omits remediation steps."}}

Question: {question}
Response: {response}"""

def judge_response(question, response, retries=3):
    for attempt in range(retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                max_tokens=200,
                messages=[
                    {"role": "user", "content": JUDGE_PROMPT.format(
                        question=question,
                        response=response[:2000]
                    )}
                ]
            )
            raw = completion.choices[0].message.content.strip()
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f"  JSON parse failed on attempt {attempt+1}, retrying...")
            time.sleep(1)
        except Exception as e:
            print(f"  API error on attempt {attempt+1}: {e}")
            time.sleep(2)
    return {"score": None, "reason": "failed after retries"}


def evaluate_with_judge(predictions, rows, label=""):
    print(f"\nRunning LLM judge on {label} predictions...")
    results = []
    for i, (pred, ex) in enumerate(zip(predictions, rows)):
        result = judge_response(ex["user"], pred)
        results.append(result)
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  [{i+1}/{len(predictions)}] score: {result.get('score')} | {result.get('reason', '')[:80]}")
        time.sleep(0.3)

    valid_scores = [r["score"] for r in results if r["score"] is not None]
    mean_score = sum(valid_scores) / len(valid_scores) if valid_scores else 0
    print(f"\n{label} Judge Score (mean): {mean_score:.3f} (n={len(valid_scores)})")
    return results, mean_score


base_judge_results, base_judge_mean = evaluate_with_judge(base_predictions, rows, label="Base Gemma E4B")
ft_judge_results,   ft_judge_mean   = evaluate_with_judge(ft_predictions,   rows, label="Fine-tuned Gemma E4B")


Running LLM judge on Base Gemma E4B predictions...
  [1/50] score: 5 | Response is technically accurate, complete, well-structured, and directly addres
  [10/50] score: 4 | Response accurately outlines the primary threat vectors associated with Intellig
  [20/50] score: 4 | Response accurately discusses the risks associated with shared memory segments a
  [30/50] score: 4 | Response provides a solid overview of purple teaming in the context of software 
  [40/50] score: 4 | Response accurately discusses the role of deception technologies in detecting wi
  [50/50] score: 4 | The response provides a well-structured overview of victimology analysis and its

Base Gemma E4B Judge Score (mean): 3.900 (n=50)

Running LLM judge on Fine-tuned Gemma E4B predictions...
  [1/50] score: 5 | Response accurately describes how adversary emulation can expose gaps in threat 
  [10/50] score: 4 | Response accurately identifies relevant threat models and associated techniques 
  [20/50] score: 4 | Respon

In [17]:
FOUNDATION_MODEL_ID = "fdtn-ai/Foundation-Sec-8B"

print(f"\nLoading {FOUNDATION_MODEL_ID}...")
foundation_tokenizer = AutoTokenizer.from_pretrained(FOUNDATION_MODEL_ID)
if foundation_tokenizer.pad_token is None:
    foundation_tokenizer.pad_token = foundation_tokenizer.eos_token

foundation_model = AutoModelForCausalLM.from_pretrained(
    FOUNDATION_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
foundation_model.eval()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

def build_foundation_prompt(example):
    return (
        f"### System:\n{example['system']}\n\n"
        f"### User:\n{example['user']}\n\n"
        f"### Assistant:\n"
    )

foundation_predictions = []
for i, ex in enumerate(rows):
    prompt = build_foundation_prompt(ex)
    inputs = foundation_tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(foundation_model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = foundation_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=foundation_tokenizer.pad_token_id,
            eos_token_id=foundation_tokenizer.eos_token_id,
        )
    pred = foundation_tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    foundation_predictions.append(pred)
    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(rows)}] input : {ex['user'][:60]}...")
        print(f"           output: {pred[:60]}...")

del foundation_model
torch.cuda.empty_cache()

# Judge evaluation for Foundation-Sec-8B
foundation_judge_results, foundation_judge_mean = evaluate_with_judge(
    foundation_predictions, rows, label="Foundation-Sec-8B"
)


Loading fdtn-ai/Foundation-Sec-8B...


config.json:   0%|          | 0.00/840 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/630 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

VRAM used: 64.88 GB


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[1/50] input : In what situations would adversary emulation reveal gaps in ...
           output: Adversary emulation is a proactive security exercise where d...
[10/50] input : Which situation-specific threat models best capture the atta...
           output: The MITRE ATT&CK framework provides a comprehensive matrix o...
[20/50] input : What are the techniques to detect and recover shared memory ...
           output: To detect shared memory segments, you can use tools like `ls...
[30/50] input : What are the techniques for developing purple team methodolo...
           output: Purple teaming for software supply chain security involves a...
[40/50] input : In which circumstances could deception technologies (honeypo...
           output: Deception technologies can be highly effective in detecting ...
[50/50] input : What techniques can defenders use to correlate campaigns thr...
           output: Defenders can correlate campaigns through victimology patter...

Running LLM judge on F

In [18]:
foundation_rouge = [rouge.score(ref, pred)["rougeL"].fmeasure
                    for ref, pred in zip(references, foundation_predictions)]

_, _, foundation_bert = bert_score_fn(
    foundation_predictions, references,
    lang="en", model_type="roberta-large",
    device="cpu", use_fast_tokenizer=False, verbose=False
)

print("\n" + "="*65)
print(f"{'Metric':<20} {'Base E4B':>12} {'Fine-tuned E4B':>15} {'Foundation-8B':>14}")
print("="*65)
print(f"{'ROUGE-L F1':<20} {sum(base_rouge)/len(base_rouge):>12.4f} {sum(ft_rouge)/len(ft_rouge):>15.4f} {sum(foundation_rouge)/len(foundation_rouge):>14.4f}")
print(f"{'BERTScore F1':<20} {base_bert.mean().item():>12.4f} {ft_bert.mean().item():>15.4f} {foundation_bert.mean().item():>14.4f}")
print(f"{'LLM Judge (1-5)':<20} {base_judge_mean:>12.3f} {ft_judge_mean:>15.3f} {foundation_judge_mean:>14.3f}")
print("="*65)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Metric                   Base E4B  Fine-tuned E4B  Foundation-8B
ROUGE-L F1                 0.1284          0.1857         0.1481
BERTScore F1               0.8284          0.8628         0.8341
LLM Judge (1-5)             3.900           4.180          3.780
